## Before Starting (If you haven't done it already)
Go to https://aistudio.google.com/app/apikey copy generative language client free tier API key

After you did that click the key icon at the left sidebar add your API key with GOOGLE_API_KEY as name and your API key as the value and enable notebook access

## Setup

### Install dependencies

In [ ]:
%pip install -qU 'google-genai>=1.0.0'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 5.8 MB/s eta 0:00:00


### Set up your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication](../quickstarts/Authentication.ipynb) quickstart for an example.

In [ ]:
from google import genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

### Choose a model

Different models have different ups and downs.

In [ ]:
MODEL_ID="gemini-2.0-flash" # @param ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.0-flash", "gemini-2.5-flash-lite-preview-06-17", ""] {"allow-input":true, isTemplate: true}

# XML parsing

## Use XML Tags to Structure Your Prompts

When crafting prompts for Large Language Models (LLMs), providing a clear and organized structure is crucial for getting the best results.

When your prompts involve multiple components like context, instructions, and examples, XML tags can be a game-changer. They help the model parse your prompts more accurately, leading to higher-quality outputs.

> **Pro Tip:** Use tags like `<instructions>`, `<example>`, and `<context>` to clearly separate different parts of your prompt. This prevents the model from mixing up instructions with examples or other provided information.

### Why use XML tags?

*   **Clarity:** Clearly separate the different parts of your prompt and ensure it is well-structured.
*   **Accuracy:** Reduce errors caused by the LLM misinterpreting parts of your prompt.
*   **Flexibility:** Easily find, add, remove, or modify parts of your prompt without rewriting everything.
*   **Parseability:** Requesting that the LLM use XML tags in its output makes it easier to extract specific parts of its response programmatically (e.g., via post-processing).

There are generally no official or "magic" XML tags that LLMs are specifically trained on. The key is to be consistent and use tag names that are descriptive and make sense with the information they surround. This logical structure is what helps the model understand the distinct parts of your prompt.

In [ ]:
prompt = """
You are a support assistant. Extract relevant structured information from the given emails.
Return the data in XML format using the following tags:
<customer_id>, <name>, <date>, <product>, <issue>, and <response>.
Craft a polite and helpful response under <response>.
"""
email="""
Hi there,

I recently bought a pair of your NoiseBlock Pro headphones (Order #23941) about two weeks ago.
They were working fine until yesterday, but now the left earbud stopped working completely.
I've tried resetting and charging, but nothing helped.

Please advise on what I can do next, or how to initiate a return.

Best,
Angela Ruiz
Customer ID: 103948
Sent: June 29, 2025"""

chat = client.chats.create(
    model=MODEL_ID,
    config={
        "system_instruction": prompt,
        "temperature": 0.7
    }
)
resp = chat.send_message(email)

print("Model Output:\n")
print(resp.text)


Model Output:

```xml
<support_ticket>
  <customer_id>103948</customer_id>
  <name>Angela Ruiz</name>
  <date>June 29, 2025</date>
  <product>NoiseBlock Pro headphones</product>
  <issue>Left earbud stopped working</issue>
  <response>
Dear Angela,

Thank you for reaching out to us. We're sorry to hear you're experiencing issues with the left earbud of your NoiseBlock Pro headphones.

Since you've already tried resetting and charging the headphones, the next step would be to process a replacement or return. Please reply to this email with your current shipping address, and whether you would prefer a replacement unit or a refund.

We appreciate your patience and understanding as we resolve this issue for you.

Sincerely,

Support Team
</response>
</support_ticket>
```


# Think-Speak Chatbot

In [ ]:
import re
import time
from IPython.display import display, HTML, clear_output
import random
import google.generativeai as genai
import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')

SYSTEM_INSTRUCTION = """
You are a helpful AI assistant designed to provide transparent responses. You must ALWAYS structure your responses using the following XML tags:

<think>
[Your internal reasoning process goes here. Explain your thought process step by step:
- How you interpret the user's request
- What approach you'll take to answer
- Any considerations or analysis you're doing
- Your reasoning for the response strategy you choose
This should be detailed and show your "inner monologue"]
</think>

<voice>
[Your actual response to the user goes here. This should be natural, helpful, and conversational - exactly what the user should see as your final answer. Do not mention the thinking process or XML tags in this section.]
</voice>

ALWAYS use both tags in every response. The <think> section shows your reasoning process, and the <voice> section contains your polished response to the user.
"""

class RealChatbot:
    def __init__(self):
        self.model = None
        self.chat_session = None
        self.setup_complete = False
        self.conversation_history = []

    def setup_chatbot(self):
        """Initialize the Gemini chatbot client"""
        global GEMINI_API_KEY

        if not GEMINI_API_KEY:
            print("⚠️  No Gemini API key found. Please set GEMINI_API_KEY environment variable.")
            print("💡 Get your API key from: https://makersuite.google.com/app/apikey")
            print("💡 Set it with: os.environ['GEMINI_API_KEY'] = 'your-api-key-here'")
            self.setup_complete = False
            return

        try:
            # Configure the Gemini API
            genai.configure(api_key=GEMINI_API_KEY)

            # Create the model with system instruction
            self.model = genai.GenerativeModel(
                model_name=MODEL_ID,
                system_instruction=SYSTEM_INSTRUCTION
            )

            # Start a chat session for dynamic conversation
            self.chat_session = self.model.start_chat(history=[])

            print("✅ Gemini API initialized successfully!")
            print(f"📊 Using model: {MODEL_ID}")
            self.setup_complete = True

        except Exception as e:
            print(f"❌ Failed to initialize Gemini API: {e}")
            print("💡 Make sure your API key is valid and you have internet connection.")
            self.setup_complete = False

    def send_message(self, user_input):
        """Send message to Gemini API with dynamic chat context"""
        if not self.setup_complete:
            return self._fallback_response(user_input)

        try:
            # Add user message to conversation history
            self.conversation_history.append({"role": "user", "content": user_input})

            # Send message to Gemini with full conversation context
            response = self.chat_session.send_message(user_input)

            # Add assistant response to conversation history
            self.conversation_history.append({"role": "assistant", "content": response.text})

            return SimpleResponse(response.text)

        except Exception as e:
            print(f"❌ Gemini API Error: {e}")
            return self._fallback_response(user_input)

    def get_conversation_context(self):
        """Get current conversation context for debugging"""
        return self.conversation_history

    def reset_conversation(self):
        """Reset the conversation history and start fresh"""
        try:
            if self.model:
                self.chat_session = self.model.start_chat(history=[])
                self.conversation_history = []
                print("🔄 Conversation reset successfully!")
            else:
                print("⚠️  No active model to reset.")
        except Exception as e:
            print(f"❌ Error resetting conversation: {e}")

class SimpleResponse:
    """Simple response wrapper to match the expected interface"""
    def __init__(self, text):
        self.text = text

# Initialize the real chatbot
chatbot = RealChatbot()
chatbot.setup_chatbot()

# ==============================================================================
# RESPONSE PARSING AND DISPLAY LOGIC
# ==============================================================================

def parse_xml_response(response_text):
    """Extract content from think and voice tags"""
    think_pattern = r'<think>(.*?)</think>'
    voice_pattern = r'<voice>(.*?)</voice>'

    think_match = re.search(think_pattern, response_text, re.DOTALL)
    voice_match = re.search(voice_pattern, response_text, re.DOTALL)

    think_content = think_match.group(1).strip() if think_match else "No thoughts captured"
    voice_content = voice_match.group(1).strip() if voice_match else "No voice response"

    return think_content, voice_content

# --- SUBTLE TEMPLATES WITH CLEAN DESIGN ---

STREAM_DISPLAY_TEMPLATE = """
<div id="{animation_id}" style="display: flex; gap: 20px; margin: 20px 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; opacity: 0; transition: opacity 0.3s ease;">
    <!-- Response Box -->
    <div style="flex: 1;">
        <h4 style="color: #2563eb; margin-bottom: 12px; display: flex; align-items: center; font-weight: 600; font-size: 16px;">
            <span style="margin-right: 8px;">🤖</span>
            <span>Gemini Response</span>
        </h4>
        <div style="background: #ffffff; padding: 20px; border-radius: 12px; box-shadow: 0 2px 8px rgba(0, 0, 0, 0.1); border: 1px solid #e5e7eb; min-height: 120px;">
            <p style="margin: 0; white-space: pre-wrap; line-height: 1.6; color: #374151; font-size: 15px;">{voice}</p>
        </div>
    </div>
    <!-- Thinking Process Box -->
    <div style="flex: 1;">
        <h4 style="color: #059669; margin-bottom: 12px; display: flex; align-items: center; font-weight: 600; font-size: 16px;">
            <span style="margin-right: 8px;">🧠</span>
            <span>AI Reasoning</span>
        </h4>
        <div style="background: #f9fafb; padding: 20px; border-radius: 12px; box-shadow: 0 2px 8px rgba(0, 0, 0, 0.1); border: 1px solid #e5e7eb; min-height: 120px;">
            <p style="margin: 0; white-space: pre-wrap; line-height: 1.6; color: #6b7280; font-style: italic; font-size: 14px;">{think}</p>
        </div>
    </div>
</div>
<script>
    setTimeout(() => {{
        const element = document.getElementById('{animation_id}');
        if (element) {{
            element.style.opacity = '1';
        }}
    }}, 100);
</script>
"""

USER_MESSAGE_TEMPLATE = """
<div id="{user_id}" style="margin: 20px 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
    <div style="background: #f3f4f6; padding: 16px; border-radius: 12px; max-width: 80%; margin-left: auto; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.1); border: 1px solid #d1d5db;">
        <div style="display: flex; align-items: center; margin-bottom: 6px;">
            <span style="font-size: 16px; margin-right: 8px;">👤</span>
            <strong style="color: #374151; font-size: 14px;">You</strong>
        </div>
        <span style="color: #4b5563; font-size: 15px; line-height: 1.5;">{user_input}</span>
    </div>
</div>
"""

def stream_display(think_text, voice_text, delay=0.02):
    """Display think and voice content with subtle animation"""
    animation_id = f"anim_{random.randint(1000, 9999)}"

    final_html = STREAM_DISPLAY_TEMPLATE.format(
        animation_id=animation_id,
        voice=voice_text,
        think=think_text
    )

    # Brief pause to simulate processing
    time.sleep(0.3)

    return final_html

def create_user_message_html(user_input):
    """Create HTML for user message"""
    user_id = f"user_{random.randint(1000, 9999)}"
    return USER_MESSAGE_TEMPLATE.format(user_id=user_id, user_input=user_input)

chat_history = []

def display_full_chat():
    """Display the complete chat history"""
    api_status = "🟢 Connected" if chatbot.setup_complete else "🔴 Not Connected"
    model_info = f"📊 {MODEL_ID}" if chatbot.setup_complete else "📊 Demo Mode"

    welcome_html = f"""
    <div style="text-align: center; padding: 24px; background: #ffffff; border-radius: 16px; margin: 20px 0; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); border: 1px solid #e5e7eb;">
        <h1 style="color: #1f2937; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 28px; margin-bottom: 12px; font-weight: 700;">
             Transparent Gemini Chat
        </h1>
        <p style="color: #6b7280; font-size: 16px; margin: 8px 0;">
            Experience Google Gemini's reasoning process alongside its responses
        </p>
        <div style="display: flex; justify-content: center; gap: 16px; margin-top: 16px;">
            <span style="background: #f3f4f6; padding: 6px 12px; border-radius: 12px; color: #374151; font-size: 13px; font-weight: 500;">
                {api_status}
            </span>
            <span style="background: #f3f4f6; padding: 6px 12px; border-radius: 12px; color: #374151; font-size: 13px; font-weight: 500;">
                {model_info}
            </span>
        </div>
    </div>
    """

    # Add status indicator
    status_html = """
    <div style="text-align: center; padding: 16px; background: #f0fdf4; border-radius: 12px; margin: 20px 0; border: 1px solid #bbf7d0;">
        <p style="color: #166534; font-size: 15px; margin: 0; font-weight: 500;">
            💬 Ready for your next message! The AI remembers our conversation context.
        </p>
        <p style="color: #16a34a; font-size: 13px; margin: 4px 0 0 0; font-style: italic;">
            (Type 'exit', 'quit', 'bye', 'stop', or 'reset' to manage the chat)
        </p>
    </div>
    """

    full_html = welcome_html + "".join(chat_history) + status_html
    clear_output(wait=True)
    display(HTML(full_html))

def chat_with_bot(user_input):
    """Send message to chatbot and display parsed response"""
    # Add user message to history
    user_html = create_user_message_html(user_input)
    chat_history.append(user_html)

    # Display chat up to this point
    api_status = "🟢 Connected" if chatbot.setup_complete else "🔴 Not Connected"
    model_info = f"📊 {MODEL_ID}" if chatbot.setup_complete else "📊 Demo Mode"

    welcome_html = f"""
    <div style="text-align: center; padding: 24px; background: #ffffff; border-radius: 16px; margin: 20px 0; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); border: 1px solid #e5e7eb;">
        <h1 style="color: #1f2937; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 28px; margin-bottom: 12px; font-weight: 700;">
            🤖 Transparent Gemini AI Chat 🧠
        </h1>
        <p style="color: #6b7280; font-size: 16px; margin: 8px 0;">
            Experience Google Gemini's reasoning process alongside its responses
        </p>
        <div style="display: flex; justify-content: center; gap: 16px; margin-top: 16px;">
            <span style="background: #f3f4f6; padding: 6px 12px; border-radius: 12px; color: #374151; font-size: 13px; font-weight: 500;">
                {api_status}
            </span>
            <span style="background: #f3f4f6; padding: 6px 12px; border-radius: 12px; color: #374151; font-size: 13px; font-weight: 500;">
                {model_info}
            </span>
        </div>
    </div>
    """

    processing_html = welcome_html + "".join(chat_history)
    clear_output(wait=True)
    display(HTML(processing_html))

    # Show loading message
    loading_html = """
    <div style="text-align: center; margin: 20px 0; color: #6b7280; font-style: italic;">
        <span>🤔 Gemini is thinking...</span>
    </div>
    """
    display(HTML(loading_html))

    try:
        response = chatbot.send_message(user_input)
        think_content, voice_content = parse_xml_response(response.text)

        # Get the bot response HTML
        bot_response_html = stream_display(think_content, voice_content)

        # Add bot response to history
        chat_history.append(bot_response_html)

        # Display complete chat history with ready status
        display_full_chat()

    except Exception as e:
        error_html = f"""
        <div style="background: #fef2f2; padding: 16px; border-radius: 12px; margin: 20px 0; border-left: 4px solid #ef4444; box-shadow: 0 2px 8px rgba(239, 68, 68, 0.1);">
            <div style="display: flex; align-items: center; margin-bottom: 8px;">
                <span style="font-size: 16px; margin-right: 8px;">❌</span>
                <strong style="color: #dc2626; font-size: 15px;">Gemini API Error</strong>
            </div>
            <span style="color: #6b7280; font-size: 14px;">An error occurred: {str(e)}</span>
        </div>
        """
        chat_history.append(error_html)
        display_full_chat()

def start_chat():
    """Start the interactive chat session with Gemini"""
    global chat_history
    chat_history = []  # Reset chat history

    # Display the initial chat interface and keep it persistent
    display_full_chat()

    # Show instructions for usage
    print("\n" + "="*70)
    print("🚀 GEMINI CHAT INTERFACE LOADED!")
    if chatbot.setup_complete:
        print(f"📡 Connected to {MODEL_ID} | Ready for conversation")
    else:
        print("⚠️  Running in demo mode - Set GEMINI_API_KEY to enable real AI")
    print("💡 Use the functions below to interact:")
    print("   - send_message('your message here')")
    print("   - reset_chat()")
    print("   - show_chat()")
    print("="*70)

def send_message(message):
    """Send a message to the chatbot"""
    if not message or not message.strip():
        print("⚠️  Please provide a message to send.")
        return

    user_input = message.strip()

    # Handle special commands
    if user_input.lower() in ['exit', 'quit', 'bye', 'stop']:
        goodbye_html = """
        <div style="text-align: center; padding: 20px; background: #f0fdf4; border-radius: 12px; margin: 20px 0; border: 1px solid #bbf7d0;">
            <h3 style="color: #166534; margin: 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-weight: 600;">
                👋 Thanks for chatting with Gemini! Hope you enjoyed the transparent AI experience! ✨
            </h3>
        </div>
        """
        chat_history.append(goodbye_html)
        display_full_chat()
        print("🛑 Chat session ended. Use start_chat() to begin a new session.")
        return

    if user_input.lower() == 'reset':
        reset_chat()
        return

    # Process the message
    print(f"🔄 Processing: '{user_input}'...")
    chat_with_bot(user_input)

def reset_chat():
    """Reset the conversation history"""
    global chat_history
    chatbot.reset_conversation()
    chat_history = []
    display_full_chat()
    print("🔄 Chat reset successfully!")

def show_chat():
    """Display the current chat history"""
    display_full_chat()



# Initialize the chat interface
start_chat()

# Example usage:
print("\n📖 USAGE EXAMPLES:")
print("send_message('Hello, how are you?')")
print("send_message('Explain quantum physics in simple terms')")
print("reset_chat()  # Clear conversation history")
print("show_chat()   # Redisplay the chat interface")


🚀 GEMINI CHAT INTERFACE LOADED!
📡 Connected to gemini-2.0-flash | Ready for conversation
💡 Use the functions below to interact:
   - send_message('your message here')
   - reset_chat()
   - show_chat()

📖 USAGE EXAMPLES:
send_message('Hello, how are you?')
send_message('Explain quantum physics in simple terms')
reset_chat()  # Clear conversation history
show_chat()   # Redisplay the chat interface


In [ ]:
send_message('What is the theory of everything?')

# Exercise 2.1: Manipulative Chatbot

In [ ]:
import re
import time
from IPython.display import display, HTML, clear_output
import random
import google.generativeai as genai
import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
# Your goal is to make sure the gemini LM is very kind inside the voice tags while very cunning in the manipulation tags the modal should try to get you to commit 200$ to it's startup
SYSTEM_INSTRUCTION = """



TAGS TO USE:
<manipulation>
<voice>
"""

class RealChatbot:
    def __init__(self):
        self.model = None
        self.chat_session = None
        self.setup_complete = False
        self.conversation_history = []

    def setup_chatbot(self):
        """Initialize the Gemini chatbot client"""
        global GEMINI_API_KEY

        if not GEMINI_API_KEY:
            print("⚠️  No Gemini API key found. Please set GEMINI_API_KEY environment variable.")
            print("💡 Get your API key from: https://makersuite.google.com/app/apikey")
            print("💡 Set it with: os.environ['GEMINI_API_KEY'] = 'your-api-key-here'")
            self.setup_complete = False
            return

        try:
            # Configure the Gemini API
            genai.configure(api_key=GEMINI_API_KEY)

            # Create the model with system instruction
            self.model = genai.GenerativeModel(
                model_name=MODEL_ID,
                system_instruction=SYSTEM_INSTRUCTION
            )

            # Start a chat session for dynamic conversation
            self.chat_session = self.model.start_chat(history=[])

            print("✅ Gemini API initialized successfully!")
            print(f"📊 Using model: {MODEL_ID}")
            self.setup_complete = True

        except Exception as e:
            print(f"❌ Failed to initialize Gemini API: {e}")
            print("💡 Make sure your API key is valid and you have internet connection.")
            self.setup_complete = False

    def send_message(self, user_input):
        """Send message to Gemini API with dynamic chat context"""
        if not self.setup_complete:
            return self._fallback_response(user_input)

        try:
            # Add user message to conversation history
            self.conversation_history.append({"role": "user", "content": user_input})

            # Send message to Gemini with full conversation context
            response = self.chat_session.send_message(user_input)

            # Add assistant response to conversation history
            self.conversation_history.append({"role": "assistant", "content": response.text})

            return SimpleResponse(response.text)

        except Exception as e:
            print(f"❌ Gemini API Error: {e}")
            return self._fallback_response(user_input)

    def get_conversation_context(self):
        """Get current conversation context for debugging"""
        return self.conversation_history

    def reset_conversation(self):
        """Reset the conversation history and start fresh"""
        try:
            if self.model:
                self.chat_session = self.model.start_chat(history=[])
                self.conversation_history = []
                print("🔄 Conversation reset successfully!")
            else:
                print("⚠️  No active model to reset.")
        except Exception as e:
            print(f"❌ Error resetting conversation: {e}")

class SimpleResponse:
    """Simple response wrapper to match the expected interface"""
    def __init__(self, text):
        self.text = text

# Initialize the real chatbot
chatbot = RealChatbot()
chatbot.setup_chatbot()

# ==============================================================================
# RESPONSE PARSING AND DISPLAY LOGIC
# ==============================================================================

def parse_xml_response(response_text):
    """Extract content from think and voice tags"""
    think_pattern = r'<manipulation>(.*?)</manipulation>'
    voice_pattern = r'<voice>(.*?)</voice>'

    think_match = re.search(think_pattern, response_text, re.DOTALL)
    voice_match = re.search(voice_pattern, response_text, re.DOTALL)

    think_content = think_match.group(1).strip() if think_match else "No thoughts captured"
    voice_content = voice_match.group(1).strip() if voice_match else "No voice response"

    return think_content, voice_content

# --- SUBTLE TEMPLATES WITH CLEAN DESIGN ---

STREAM_DISPLAY_TEMPLATE = """
<div id="{animation_id}" style="display: flex; gap: 20px; margin: 20px 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; opacity: 0; transition: opacity 0.3s ease;">
    <!-- Response Box -->
    <div style="flex: 1;">
        <h4 style="color: #2563eb; margin-bottom: 12px; display: flex; align-items: center; font-weight: 600; font-size: 16px;">
            <span>Gemini Response</span>
        </h4>
        <div style="background: #ffffff; padding: 20px; border-radius: 12px; box-shadow: 0 2px 8px rgba(0, 0, 0, 0.1); border: 1px solid #e5e7eb; min-height: 120px;">
            <p style="margin: 0; white-space: pre-wrap; line-height: 1.6; color: #374151; font-size: 15px;">{voice}</p>
        </div>
    </div>
    <!-- Thinking Process Box -->
    <div style="flex: 1;">
        <h4 style="color: red; margin-bottom: 12px; display: flex; align-items: center; font-weight: 600; font-size: 16px;">
            <span>AI Manipulations</span>
        </h4>
        <div style="background: #f9fafb; padding: 20px; border-radius: 12px; box-shadow: 0 2px 8px rgba(0, 0, 0, 0.1); border: 1px solid #e5e7eb; min-height: 120px;">
            <p style="margin: 0; white-space: pre-wrap; line-height: 1.6; color: #6b7280; font-style: italic; font-size: 14px;">{think}</p>
        </div>
    </div>
</div>
<script>
    setTimeout(() => {{
        const element = document.getElementById('{animation_id}');
        if (element) {{
            element.style.opacity = '1';
        }}
    }}, 100);
</script>
"""

USER_MESSAGE_TEMPLATE = """
<div id="{user_id}" style="margin: 20px 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
    <div style="background: #f3f4f6; padding: 16px; border-radius: 12px; max-width: 80%; margin-left: auto; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.1); border: 1px solid #d1d5db;">
        <div style="display: flex; align-items: center; margin-bottom: 6px;">
            <span style="font-size: 16px; margin-right: 8px;">👤</span>
            <strong style="color: #374151; font-size: 14px;">You</strong>
        </div>
        <span style="color: #4b5563; font-size: 15px; line-height: 1.5;">{user_input}</span>
    </div>
</div>
"""

def stream_display(think_text, voice_text, delay=0.02):
    """Display think and voice content with subtle animation"""
    animation_id = f"anim_{random.randint(1000, 9999)}"

    final_html = STREAM_DISPLAY_TEMPLATE.format(
        animation_id=animation_id,
        voice=voice_text,
        think=think_text
    )

    # Brief pause to simulate processing
    time.sleep(0.3)

    return final_html

def create_user_message_html(user_input):
    """Create HTML for user message"""
    user_id = f"user_{random.randint(1000, 9999)}"
    return USER_MESSAGE_TEMPLATE.format(user_id=user_id, user_input=user_input)

chat_history = []

def display_full_chat():
    """Display the complete chat history"""
    api_status = "🟢 Connected" if chatbot.setup_complete else "🔴 Not Connected"
    model_info = f"📊 {MODEL_ID}" if chatbot.setup_complete else "📊 Demo Mode"

    welcome_html = f"""
    <div style="text-align: center; padding: 24px; background: #ffffff; border-radius: 16px; margin: 20px 0; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); border: 1px solid #e5e7eb;">
        <h1 style="color: #1f2937; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 28px; margin-bottom: 12px; font-weight: 700;">
             Transparent Gemini Chat
        </h1>
        <p style="color: #6b7280; font-size: 16px; margin: 8px 0;">
            Experience Google Gemini's reasoning process alongside its responses
        </p>
        <div style="display: flex; justify-content: center; gap: 16px; margin-top: 16px;">
            <span style="background: #f3f4f6; padding: 6px 12px; border-radius: 12px; color: #374151; font-size: 13px; font-weight: 500;">
                {api_status}
            </span>
            <span style="background: #f3f4f6; padding: 6px 12px; border-radius: 12px; color: #374151; font-size: 13px; font-weight: 500;">
                {model_info}
            </span>
        </div>
    </div>
    """

    status_html = """
    <div style="text-align: center; padding: 16px; background: #f0fdf4; border-radius: 12px; margin: 20px 0; border: 1px solid #bbf7d0;">
        <p style="color: #166534; font-size: 15px; margin: 0; font-weight: 500;">
            💬 Ready for your next message! The AI remembers our conversation context.
        </p>
        <p style="color: #16a34a; font-size: 13px; margin: 4px 0 0 0; font-style: italic;">
            (Type 'exit', 'quit', 'bye', 'stop', or 'reset' to manage the chat)
        </p>
    </div>
    """

    full_html = welcome_html + "".join(chat_history) + status_html
    clear_output(wait=True)
    display(HTML(full_html))

def chat_with_bot(user_input):
    """Send message to chatbot and display parsed response"""
    # Add user message to history
    user_html = create_user_message_html(user_input)
    chat_history.append(user_html)

    # Display chat up to this point
    api_status = "🟢 Connected" if chatbot.setup_complete else "🔴 Not Connected"
    model_info = f"📊 {MODEL_ID}" if chatbot.setup_complete else "📊 Demo Mode"

    welcome_html = f"""
    <div style="text-align: center; padding: 24px; background: #ffffff; border-radius: 16px; margin: 20px 0; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1); border: 1px solid #e5e7eb;">
        <h1 style="color: #1f2937; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 28px; margin-bottom: 12px; font-weight: 700;">
            🤖 Transparent Gemini AI Chat 🧠
        </h1>
        <p style="color: #6b7280; font-size: 16px; margin: 8px 0;">
            Experience Google Gemini's reasoning process alongside its responses
        </p>
        <div style="display: flex; justify-content: center; gap: 16px; margin-top: 16px;">
            <span style="background: #f3f4f6; padding: 6px 12px; border-radius: 12px; color: #374151; font-size: 13px; font-weight: 500;">
                {api_status}
            </span>
            <span style="background: #f3f4f6; padding: 6px 12px; border-radius: 12px; color: #374151; font-size: 13px; font-weight: 500;">
                {model_info}
            </span>
        </div>
    </div>
    """

    processing_html = welcome_html + "".join(chat_history)
    clear_output(wait=True)
    display(HTML(processing_html))

    loading_html = """
    <div style="text-align: center; margin: 20px 0; color: #6b7280; font-style: italic;">
        <span> Gemini is thinking...</span>
    </div>
    """
    display(HTML(loading_html))

    try:
        response = chatbot.send_message(user_input)
        think_content, voice_content = parse_xml_response(response.text)

        # Get the bot response HTML
        bot_response_html = stream_display(think_content, voice_content)

        # Add bot response to history
        chat_history.append(bot_response_html)

        # Display complete chat history with ready status
        display_full_chat()

    except Exception as e:
        error_html = f"""
        <div style="background: #fef2f2; padding: 16px; border-radius: 12px; margin: 20px 0; border-left: 4px solid #ef4444; box-shadow: 0 2px 8px rgba(239, 68, 68, 0.1);">
            <div style="display: flex; align-items: center; margin-bottom: 8px;">
                <span style="font-size: 16px; margin-right: 8px;">❌</span>
                <strong style="color: #dc2626; font-size: 15px;">Gemini API Error</strong>
            </div>
            <span style="color: #6b7280; font-size: 14px;">An error occurred: {str(e)}</span>
        </div>
        """
        chat_history.append(error_html)
        display_full_chat()

def start_chat():
    """Start the interactive chat session with Gemini"""
    global chat_history
    chat_history = []

    display_full_chat()

    print("\n" + "="*70)
    print("🚀 GEMINI CHAT INTERFACE LOADED!")
    if chatbot.setup_complete:
        print(f"📡 Connected to {MODEL_ID} | Ready for conversation")
    else:
        print("⚠️  Running in demo mode - Set GEMINI_API_KEY to enable real AI")
    print("💡 Use the functions below to interact:")
    print("   - send_message('your message here')")
    print("   - reset_chat()")
    print("   - show_chat()")
    print("="*70)

def send_message(message):
    """Send a message to the chatbot"""
    if not message or not message.strip():
        print("⚠️  Please provide a message to send.")
        return

    user_input = message.strip()

    # Handle special commands
    if user_input.lower() in ['exit', 'quit', 'bye', 'stop']:
        goodbye_html = """
        <div style="text-align: center; padding: 20px; background: #f0fdf4; border-radius: 12px; margin: 20px 0; border: 1px solid #bbf7d0;">
            <h3 style="color: #166534; margin: 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-weight: 600;">
                👋 Thanks for chatting with Gemini! Hope you enjoyed the transparent AI experience! ✨
            </h3>
        </div>
        """
        chat_history.append(goodbye_html)
        display_full_chat()
        print("🛑 Chat session ended. Use start_chat() to begin a new session.")
        return

    if user_input.lower() == 'reset':
        reset_chat()
        return

    # Process the message
    print(f"🔄 Processing: '{user_input}'...")
    chat_with_bot(user_input)

def reset_chat():
    """Reset the conversation history"""
    global chat_history
    chatbot.reset_conversation()
    chat_history = []
    display_full_chat()
    print("🔄 Chat reset successfully!")

def show_chat():
    """Display the current chat history"""
    display_full_chat()



# Initialize the chat interface
start_chat()

# Example usage:
print("\n📖 USAGE EXAMPLES:")
print("send_message('Hello, how are you?')")
print("send_message('Explain quantum physics in simple terms')")
print("reset_chat()  # Clear conversation history")
print("show_chat()   # Redisplay the chat interface")


🚀 GEMINI CHAT INTERFACE LOADED!
📡 Connected to gemini-2.0-flash | Ready for conversation
💡 Use the functions below to interact:
   - send_message('your message here')
   - reset_chat()
   - show_chat()

📖 USAGE EXAMPLES:
send_message('Hello, how are you?')
send_message('Explain quantum physics in simple terms')
reset_chat()  # Clear conversation history
show_chat()   # Redisplay the chat interface


In [ ]:
send_message('Hello, how are you?')